## Exercise Task

Give the Model Grader more context on what a goog solution looks like

- Step 1: Update the dataset generations prompt to ask for some "solution criteria" to be included for each test case

- Step 2: Update the 'grade_by_model' prompt to include that solution criteria

In [11]:
from anthropic import Anthropic
from dotenv import load_dotenv
import os

load_dotenv()

token = os.environ["ANTHROPIC_AUTH_TOKEN"]
url = os.environ["ANTHROPIC_BASE_URL"]
model = os.environ["ANTHROPIC_DEFAULT_HAIKU_MODEL"]

client = Anthropic(
    api_key=token,
    base_url=url,
    default_headers={"Authorization": f"Bearer {token}"}
)

def add_user_message(messages, content):
    user_message = { "role": "user", "content": content }
    messages.append(user_message)

def add_assistant_message(messages, content):
    assistant_message = { "role": "assistant", "content": content }
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):

  params = {
    "model": model,
    "max_tokens": 1000,
    "messages": messages,
    "temperature": temperature,
    "stop_sequences": stop_sequences
  }

  if system:
    params["system"] = system

  message = client.messages.create(**params)
  return message.content[0].text

In [12]:
# Function to generate a new dataset
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json" or "python" or "regex",
        "solution_criteria": "Key criteria for evaluating the solution"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [13]:
# Generate the dataset and write it to 'dataset.json'
dataset = generate_dataset()
with open("exercise-dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [14]:
# Function to grade a test case + output using a model
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Criteria you should use to evaluate the solution:
<criteria>
{test_case["solution_criteria"]}
</criteria>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [15]:
# Passes a test case into Claude
def run_prompt(test_case):
    prompt = f"""
Please solve the following task:

{test_case["task"]}

* Respond only with Python, JSON, or a plain Regex
* Do not add any comments or commentary or explanation
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(messages, stop_sequences=["```"])
    return output

In [16]:
# Functions to validate the output structure
import re
import ast


def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)


In [17]:
# Function to execute a single test case and grade the output
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    syntax_score = grade_syntax(output, test_case)

    score = (model_score + syntax_score) / 2

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }

In [18]:
from statistics import mean


def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

In [19]:
with open("exercise-dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average score: 8.333333333333334


In [20]:
print(json.dumps(results, indent=2))

[
  {
    "output": "\nimport json\nimport sys\n\ndef extract_s3_bucket_ids(template_str):\n    try:\n        template = json.loads(template_str)\n    except json.JSONDecodeError:\n        return []\n    \n    resources = template.get('Resources', {})\n    s3_bucket_ids = []\n    \n    for logical_id, resource_config in resources.items():\n        if resource_config.get('Type') == 'AWS::S3::Bucket':\n            s3_bucket_ids.append(logical_id)\n    \n    return s3_bucket_ids\n\nif __name__ == '__main__':\n    template_input = sys.stdin.read()\n    result = extract_s3_bucket_ids(template_input)\n    print(json.dumps(result))\n",
    "test_case": {
      "task": "Parse an AWS CloudFormation template and extract all resource logical IDs that are of type 'AWS::S3::Bucket'",
      "format": "python",
      "solution_criteria": "Solution should return a list of logical IDs for all S3 bucket resources. Should handle JSON-formatted templates and gracefully handle missing or empty resources se